# CS781 Phase 1 — Model 2 — Generate Codabench Submission

**Inputs to attach**: (1) the tokenizer dataset, (2) model 2's training notebook output (needs its `legal-mladsa-final` folder with the final saved checkpoint), and (3) optionally, the Prepare Test Set notebook's output -- same Phase 1 Task 1 test set as model 1 used, so that snapshot is reusable here unchanged.

Needs internet + HF login only as a fallback (if no snapshot is attached). No GPU-heavy work here compared to training -- no backward pass, and KV-caching makes each of the 128 decode steps cheap after the prompt prefill.

## 1. Setup

In [ ]:
import os

## 2. Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 3. Configuration

In [ ]:
import glob, json, math, time, zipfile

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"  # verified via list_repo_files() -- see model 1's notebook history

TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 256
TEMPERATURE = 0.7
TOP_P = 0.95
GENERATION_BATCH_SIZE = 32

OUTPUT_DIR = "/kaggle/working/cs781_phase1_model2_submission"
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 4. Load Test Data

Prefers an attached snapshot from the Prepare Test Set notebook over downloading fresh -- the dataset repo notes the test file is dynamic and can change. HF login only happens in the fallback branch below, so it's never needed (and never asked for) when a snapshot is already attached, which is the normal case.

In [ ]:
def find_dirs(filename):
    return sorted({os.path.dirname(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)})

snapshot_dirs = find_dirs("test_snapshot.json")
if snapshot_dirs:
    test_file_path = os.path.join(snapshot_dirs[0], "test_snapshot.json")
    print("Using attached snapshot:", test_file_path)
    stats_path = os.path.join(snapshot_dirs[0], "test_stats.json")
    if os.path.exists(stats_path):
        print("Snapshot info:", json.load(open(stats_path)))
else:
    from huggingface_hub import login, hf_hub_download
    from kaggle_secrets import UserSecretsClient
    print("No snapshot attached -- logging in and downloading fresh (this will be whatever is live right now).")
    HF_TOKEN = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    test_file_path = hf_hub_download(
        repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN
    )
    print("Downloaded:", test_file_path)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print("Number of test examples:", len(test_data))

## 5. Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

cands = [d for d in find_dirs("tokenizer.json") if "checkpoint-" not in d and "legal-mladsa-final" not in d]
if not cands:
    raise FileNotFoundError("No tokenizer dataset found -- attach it via Add Input.")
TOKENIZER_DIR = ([d for d in cands if os.path.exists(os.path.join(d, "trained_on.json"))] or cands)[0]

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_DIR)
tokenizer.padding_side = "left"  # required for correct batched generation
print("Tokenizer:", TOKENIZER_DIR, "| vocab size:", len(tokenizer))

## 5b. Load Pre-Tokenized Test Set (if attached)

In [ ]:
tok_cache_dirs = find_dirs("test_tokenized.json")
test_token_ids = None
if tok_cache_dirs:
    tok_cache_path = os.path.join(tok_cache_dirs[0], "test_tokenized.json")
    with open(tok_cache_path) as f:
        test_token_ids = json.load(f)  # {example_id: [token ids]}
    print("Using pre-tokenized cache:", tok_cache_path, f"({len(test_token_ids)} examples)")
else:
    print("No pre-tokenized cache attached -- will tokenize prompts inline during generation.")

## 6. Define The Model

Five files, model 2's exact tested architecture, unchanged: `mla.py` (Multi-Head Latent Attention), `dsa.py` (DeepSeek Sparse Attention), `moe.py`, `mla_dsa.py` (combines MLA+DSA), `model.py`. Written to real files and imported for the same reason as every training notebook: `transformers` reads `__file__` off a model class's module.

In [ ]:
%%writefile /kaggle/working/mla.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(x, cos, sin):
    # x: (..., T, d); cos/sin: (T, d)
    return (x * cos) + (rotate_half(x) * sin)


def rope_table(T, dim, theta, device, dtype):
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2, device=device).float() / dim))
    pos = torch.arange(T, device=device).float()
    emb = torch.cat((torch.outer(pos, inv_freq),) * 2, dim=-1)
    return emb.cos().to(dtype), emb.sin().to(dtype)


class MLA(nn.Module):
    """Multi-head Latent Attention (DeepSeek-V2 style).
    Caches only the compressed latent c_kv + the shared decoupled RoPE key k_R,
    not full per-head K/V. head_dim is the per-head size of the UP-PROJECTED
    (decompressed) content part; d_rotate is appended on top of that for the
    RoPE-carrying part, so the actual attention dim per head is head_dim+d_rotate.
    """
    def __init__(self, d_model, n_head, head_dim, d_c, d_c_q, d_rotate, dropout=0.0, rope_theta=10000.0):
        super().__init__()
        self.n_head, self.head_dim, self.d_rotate = n_head, head_dim, d_rotate
        self.rope_theta = rope_theta
        self.dropout_p = dropout

        self.W_DKV = nn.Linear(d_model, d_c, bias=False)
        self.W_UK = nn.Linear(d_c, n_head * head_dim, bias=False)
        self.W_UV = nn.Linear(d_c, n_head * head_dim, bias=False)
        self.W_KR = nn.Linear(d_model, d_rotate, bias=False)  # shared across heads

        self.W_DQ = nn.Linear(d_model, d_c_q, bias=False)
        self.W_UQ = nn.Linear(d_c_q, n_head * head_dim, bias=False)
        self.W_QR = nn.Linear(d_c_q, n_head * d_rotate, bias=False)  # per head

        self.W_O = nn.Linear(n_head * head_dim, d_model, bias=False)

    def project(self, x):
        """Splits the model into the compressed-latent step (cacheable) and the
        up-projection + attention step, so DSA can sit between them and score
        against the SAME K^C/k_R this attention will use."""
        B, T, _ = x.shape
        c_kv = self.W_DKV(x)                          # (B, T, d_c)  <- what gets cached
        c_q = self.W_DQ(x)                             # (B, T, d_c_q)
        k_R = self.W_KR(x)                              # (B, T, d_rotate) <- also cached, shared across heads
        return c_kv, c_q, k_R

    def up_project(self, c_kv, c_q, k_R, past_len, device, dtype):
        B, T, _ = c_kv.shape
        H, Dh, Dr = self.n_head, self.head_dim, self.d_rotate

        K_C = self.W_UK(c_kv).view(B, T, H, Dh).transpose(1, 2)   # (B,H,T,Dh)
        V_C = self.W_UV(c_kv).view(B, T, H, Dh).transpose(1, 2)   # (B,H,T,Dh)
        Q_C = self.W_UQ(c_q).view(B, T, H, Dh).transpose(1, 2)    # (B,H,T,Dh)
        Q_R = self.W_QR(c_q).view(B, T, H, Dr).transpose(1, 2)    # (B,H,T,Dr)

        # RoPE table dtype comes from Q_R itself (the tensor actually being rotated),
        # NOT the passed-in `dtype` -- that's the pre-projection RMSNorm output, which
        # autocast can leave in fp32 (reductions like RMSNorm's .mean() are kept fp32
        # for stability) even while W_QR (a Linear/matmul) is correctly forced to fp16.
        # Using the stale pre-projection dtype here builds an fp32 cos/sin table, and
        # multiplying that into a correctly-fp16 Q_R silently promotes it back to fp32
        # -- exactly the dtype drift that caused indexer_distillation_loss's crash
        # downstream (Q ending up fp32 while indexer_scores, built the "right" way from
        # a post-projection dtype, correctly stayed fp16).
        cos, sin = rope_table(past_len + T, Dr, self.rope_theta, device, Q_R.dtype)
        cos, sin = cos[past_len:past_len + T], sin[past_len:past_len + T]
        Q_R = apply_rope(Q_R, cos, sin)
        k_R_rot = apply_rope(k_R.to(Q_R.dtype), cos, sin)          # (B,T,Dr)
        K_R = k_R_rot.unsqueeze(1).expand(B, H, T, Dr)             # broadcast to all heads

        Q = torch.cat([Q_C, Q_R], dim=-1)   # (B,H,T,Dh+Dr)
        K = torch.cat([K_C, K_R], dim=-1)   # (B,H,T,Dh+Dr)
        return Q, K, V_C

    def forward(self, x, past_kv=None, use_cache=False, key_padding_mask=None):
        """past_kv, if given: (c_kv_past, k_R_past) each (B, T_past, ·) — the
        compact cached state. NOT full per-head K/V."""
        B, T, _ = x.shape
        past_len = past_kv[0].shape[1] if past_kv is not None else 0

        c_kv, c_q, k_R = self.project(x)
        if past_kv is not None:
            full_c_kv = torch.cat([past_kv[0], c_kv], dim=1)
            full_k_R = torch.cat([past_kv[1], k_R], dim=1)
        else:
            full_c_kv, full_k_R = c_kv, k_R

        # Q only ever needed for the T new positions; K/V up-projected from the FULL cached latent
        Q, _, _ = self.up_project(c_kv, c_q, k_R, past_len, x.device, x.dtype)
        _, K_full, V_full = self._up_project_kv_only(full_c_kv, full_k_R, x.device, x.dtype)

        total_len = past_len + T
        if key_padding_mask is None and past_kv is None:
            attn_mask, is_causal = None, True
        else:
            # dtype from Q (post-projection, correctly typed after the up_project fix
            # above), not x (pre-projection, which autocast can leave in fp32) -- same
            # class of mismatch that crashed indexer_distillation_loss, fixed here too
            # since this path (padding / cached generation) wasn't exercised by the
            # training run that caught the original crash and could hit it later.
            q_pos = torch.arange(past_len, past_len + T, device=x.device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=x.device).unsqueeze(0)
            causal = (k_pos > q_pos).to(Q.dtype) * torch.finfo(Q.dtype).min
            bias = causal[None, None]
            if key_padding_mask is not None:
                pad = (1.0 - key_padding_mask[:, None, None, :].to(Q.dtype)) * torch.finfo(Q.dtype).min
                bias = bias + pad
            attn_mask, is_causal = bias, False

        out = F.scaled_dot_product_attention(
            Q, K_full, V_full, attn_mask=attn_mask, is_causal=is_causal,
            dropout_p=self.dropout_p if self.training else 0.0,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        out = self.W_O(out)
        present = (full_c_kv, full_k_R) if use_cache else None
        return out, present

    def _up_project_kv_only(self, c_kv, k_R, device, dtype):
        B, T, _ = c_kv.shape
        H, Dh, Dr = self.n_head, self.head_dim, self.d_rotate
        K_C = self.W_UK(c_kv).view(B, T, H, Dh).transpose(1, 2)
        V_C = self.W_UV(c_kv).view(B, T, H, Dh).transpose(1, 2)
        # same fix as up_project: derive RoPE dtype from a post-projection tensor
        # (K_C, correctly fp16 under autocast), not the passed-in pre-projection dtype
        cos, sin = rope_table(T, Dr, self.rope_theta, device, K_C.dtype)
        k_R_rot = apply_rope(k_R.to(K_C.dtype), cos, sin)
        K_R = k_R_rot.unsqueeze(1).expand(B, H, T, Dr)
        return None, torch.cat([K_C, K_R], dim=-1), V_C

In [ ]:
%%writefile /kaggle/working/dsa.py
import torch
import torch.nn as nn
import torch.nn.functional as F

from mla import rope_table, apply_rope


class Indexer(nn.Module):
    """Lightning indexer: cheap per-CHUNK relevance scores, feeding DSA's chunk
    selection. Keys are pooled to one summary vector per chunk BEFORE scoring
    (not scored per-token and pooled after) -- the indexer's decision is at
    chunk granularity regardless, so per-token precision here is pure waste:
    it costs O(T_block x T) for a decision that's O(T_block x num_chunks).
    Queries come from MLA's own query latent c_q (no extra projection from
    raw x); keys derive from MLA's cached c_kv; both reuse MLA's RoPE table."""
    def __init__(self, d_c_q, d_c, n_idx_heads, idx_head_dim, chunk_size, rope_theta=10000.0):
        super().__init__()
        self.H, self.Dh, self.chunk_size = n_idx_heads, idx_head_dim, chunk_size
        self.rope_theta = rope_theta
        self.W_IQ = nn.Linear(d_c_q, n_idx_heads * idx_head_dim, bias=False)
        self.W_IK = nn.Linear(d_c, idx_head_dim, bias=False)          # single shared "head"
        self.W_IW = nn.Linear(d_c_q, n_idx_heads, bias=False)          # per-query head weights

    def forward(self, c_q, c_kv, past_len=0):
        B, T, _ = c_q.shape
        Tk = c_kv.shape[1]
        cs = self.chunk_size
        num_chunks = (Tk + cs - 1) // cs
        pad = num_chunks * cs - Tk

        q = self.W_IQ(c_q).view(B, T, self.H, self.Dh)
        k = self.W_IK(c_kv)                                            # (B, Tk, Dh)
        w = self.W_IW(c_q)                                              # (B, T, H)

        cos, sin = rope_table(Tk, self.Dh, self.rope_theta, c_q.device, c_q.dtype)
        k = apply_rope(k, cos, sin)                                     # RoPE per real token position, same as before
        q = apply_rope(q.transpose(1, 2), cos[past_len:past_len + T], sin[past_len:past_len + T]).transpose(1, 2)

        if pad:
            k = F.pad(k, (0, 0, 0, pad))                                # zero-pad tail to a full chunk
        k_pooled = k.view(B, num_chunks, cs, self.Dh).mean(dim=2)       # (B, num_chunks, Dh) -- pool AFTER RoPE,
                                                                          # BEFORE scoring: the indexer only ever
                                                                          # needs a chunk-level relevance signal

        # I[b,t,c] = sum_h w[b,t,h] * relu(q[b,t,h,:] . k_pooled[b,c,:])
        scores = torch.einsum("bthd,bcd->bhtc", q, k_pooled)           # (B,H,T,num_chunks)
        scores = F.relu(scores)
        scores = torch.einsum("bhtc,bth->btc", scores, w)              # (B,T,num_chunks) -- ALREADY chunk-level
        return scores


def indexer_distillation_loss(Q, K, indexer_scores, chunk_size, causal_offset=0, sample_size=256):
    """The indexer's scores only ever feed a hard top-k, which is
    non-differentiable -- with nothing else, it never receives a training
    signal at all. DeepSeek's own recipe trains the indexer separately, via
    KL-divergence against the real (dense) attention distribution, computed
    here from Q/K already on hand. Only the QK^T + softmax is recomputed
    (no V-multiply, no backward through it -- it's a detached target).

    Computed over a random SUBSAMPLE of `sample_size` query positions, not
    all T -- this target is itself an O(T^2) dense computation (materializing
    a (B,H,T,T) tensor), and left at full T it dominates real training time
    (confirmed: ~16x the cost measured at T=1024, scaling quadratically,
    every layer, every step -- an OOM even on CPU at the real T=4096).
    Every query still gets sampled roughly every T/sample_size steps on
    average across training, which is enough for a KL-distillation signal
    that only needs to shape the indexer's general relevance judgment, not
    match it exactly at every position on every single step.

    indexer_scores is now chunk-level (B,T,num_chunks), so the token-level
    dense-attention target is pooled to the same granularity by SUMMING
    (not averaging) each chunk's token probabilities -- the correct way to
    marginalize a probability distribution over grouped outcomes. Both sides
    are then restricted to the same "whole chunk causally available" support
    that chunked_sparse_attention itself uses (a chunk counts as available
    only once its END position is valid), so the indexer is trained toward
    the same notion of availability that selection actually applies."""
    B, H, T, Dqk = Q.shape
    Tk = K.shape[2]
    device, dtype = Q.device, Q.dtype
    num_chunks = (Tk + chunk_size - 1) // chunk_size
    pad = num_chunks * chunk_size - Tk

    if T > sample_size:
        sample_idx = torch.randperm(T, device=device)[:sample_size]
        Qs = Q[:, :, sample_idx, :]
        scores_s = indexer_scores[:, sample_idx, :]
        q_abs = causal_offset + sample_idx
    else:
        Qs = Q
        scores_s = indexer_scores
        q_abs = torch.arange(causal_offset, causal_offset + T, device=device)
    Ts = Qs.shape[2]

    q_pos = q_abs.unsqueeze(1)
    k_pos = torch.arange(0, Tk, device=device).unsqueeze(0)
    causal = (k_pos > q_pos).to(dtype) * torch.finfo(dtype).min
    with torch.no_grad():
        target_logits = torch.einsum("bhtd,bhsd->bhts", Qs, K) / (Dqk ** 0.5) + causal[None, None]
        target_tok = F.softmax(target_logits, dim=-1).mean(dim=1)          # (B,Ts,Tk), averaged over heads
        if pad:
            target_tok = F.pad(target_tok, (0, pad), value=0.0)
        target = target_tok.view(B, Ts, num_chunks, chunk_size).sum(dim=-1)  # (B,Ts,num_chunks)

    real_chunk_start_pos = torch.arange(num_chunks, device=device) * chunk_size
    # A chunk only needs to START on-or-before the query to count as "available" here
    # (unlike chunked_sparse_attention's stricter end-position rule, which exists only
    # to guarantee a whole gathered chunk is safe -- this loss has no gather to protect).
    # Using that stricter rule here would mark EVERY chunk unavailable for the earliest
    # queries (even chunk 0, whose END is still in the future for query 0), giving an
    # all-(-inf) row and a NaN from log_softmax -- exactly what testing caught.
    chunk_not_ok = (real_chunk_start_pos[None, :] > q_pos)                  # (Ts,num_chunks)
    idx_logits = scores_s.masked_fill(chunk_not_ok[None], torch.finfo(scores_s.dtype).min)
    # NOTE: torch.finfo(dtype).min here, not float("-inf") -- log_softmax of a masked
    # position legitimately produces -inf, and F.kl_div computes target*input BEFORE
    # summing, so a real -inf meeting target=0 at that same position gives 0*(-inf) = NaN
    # in IEEE float, regardless of the mathematical convention that it "should" be 0.
    # A large finite negative number sidesteps this (same pattern already used for the
    # safety mask in chunked_sparse_attention, for the same underlying reason).
    idx_log_probs = F.log_softmax(idx_logits, dim=-1)

    target = target.masked_fill(chunk_not_ok[None], 0.0)
    target = target / target.sum(dim=-1, keepdim=True).clamp_min(1e-8)      # renormalize over the same support
    return F.kl_div(idx_log_probs, target, reduction="batchmean")


def chunked_sparse_attention(Q, K, V, indexer_scores, chunk_size, topk_chunks,
                              forced_local_chunks, q_block, causal_offset=0):
    """Q,K,V: (B,H,T,·). indexer_scores: (B,Tq,num_chunks) -- ALREADY chunk-level
    (the Indexer pools keys to chunk summaries before scoring, so there's no
    token-level score matrix to pool here anymore). Selection is shared across
    each block of q_block consecutive queries (see memory note above) -- attention
    itself still applies the exact per-position causal mask, so a chunk that slips
    into an early query's selected set but lies in that query's future is harmless,
    just wasted capacity, never a correctness issue."""
    B, H, Tq, Dqk = Q.shape
    Tk = K.shape[2]
    device, dtype = Q.device, Q.dtype
    num_chunks = (Tk + chunk_size - 1) // chunk_size
    pad = num_chunks * chunk_size - Tk
    if pad:
        K = F.pad(K, (0, 0, 0, pad))
        V = F.pad(V, (0, 0, 0, pad))
    K = K.view(B, H, num_chunks, chunk_size, Dqk)
    V = V.view(B, H, num_chunks, chunk_size, V.shape[-1])

    assert indexer_scores.shape[-1] == num_chunks, \
        f"indexer_scores last dim ({indexer_scores.shape[-1]}) must equal num_chunks ({num_chunks})"
    chunk_scores = indexer_scores  # already chunk-level, nothing to pool

    outputs = []
    for q0 in range(0, Tq, q_block):
        q1 = min(q0 + q_block, Tq)
        blk = q1 - q0
        last_pos = causal_offset + q1 - 1          # most-permissive causal bound for this block
        # A chunk's REAL end may be less than its padded/nominal end (the last chunk is
        # often only partially filled, e.g. mid-generation with a cache length that
        # isn't a multiple of chunk_size). Using the padded end here would wrongly mark
        # a chunk "not yet causally available" even when it holds the current token --
        # exactly the source of the incremental-vs-full-forward mismatch found in testing.
        real_chunk_end_pos = torch.clamp(
            torch.arange(num_chunks, device=device) * chunk_size + (chunk_size - 1), max=Tk - 1
        )
        causal_ok = real_chunk_end_pos <= last_pos
        # Selection must depend ONLY on the block's earliest (most restrictive) query --
        # pooling over the whole block would let a later query's data (which the
        # earlier queries can't legally see yet) influence what gets selected for them,
        # a real causality leak even though the final attention weights stay masked.
        block_score = chunk_scores[:, q0].clone()                            # (B,num_chunks)
        block_score = block_score.masked_fill(~causal_ok, float("-inf"))

        this_last_chunk = last_pos // chunk_size
        forced = torch.arange(max(0, this_last_chunk - forced_local_chunks + 1),
                               this_last_chunk + 1, device=device)
        block_score = block_score.clone()
        block_score[:, forced] = float("inf")                              # guarantee local window is kept

        k_sel = min(topk_chunks, int(causal_ok.sum().item()) if causal_ok.any() else 1)
        sel = block_score.topk(k_sel, dim=-1).indices                       # (B, k_sel)

        Dv = V.shape[-1]
        gK = torch.gather(K, 2, sel[:, None, :, None, None].expand(B, H, k_sel, chunk_size, Dqk))
        gK = gK.reshape(B, H, k_sel * chunk_size, Dqk)
        gV = torch.gather(V, 2, sel[:, None, :, None, None].expand(B, H, k_sel, chunk_size, Dv))
        gV = gV.reshape(B, H, k_sel * chunk_size, Dv)

        gathered_pos = (sel[:, :, None] * chunk_size + torch.arange(chunk_size, device=device)[None, None, :])
        gathered_pos = gathered_pos.reshape(B, k_sel * chunk_size)          # (B, gathered_len)
        q_pos = torch.arange(q0, q1, device=device) + causal_offset          # (blk,)
        safety_mask = (gathered_pos[:, None, :] > q_pos[None, :, None])     # (B, blk, gathered_len)
        bias = safety_mask[:, None, :, :].to(dtype) * torch.finfo(dtype).min

        Qb = Q[:, :, q0:q1]
        out = F.scaled_dot_product_attention(Qb, gK, gV, attn_mask=bias)
        outputs.append(out)
    return torch.cat(outputs, dim=2)

In [ ]:
%%writefile /kaggle/working/moe.py
import torch
import torch.nn as nn
import torch.nn.functional as F


class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class MoE(nn.Module):
    def __init__(self, d_model, n_routed, n_shared, top_k, expert_d_ff, aux_loss_coef=0.01):
        super().__init__()
        self.n_routed, self.top_k = n_routed, top_k
        self.aux_loss_coef = aux_loss_coef
        self.router = nn.Linear(d_model, n_routed, bias=False)
        self.routed_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_routed)])
        self.shared_experts = nn.ModuleList([Expert(d_model, expert_d_ff) for _ in range(n_shared)])
        self.last_aux_loss = None

    def forward(self, x):
        B, T, D = x.shape
        flat = x.reshape(-1, D)                                 # (N, D), N = B*T
        N = flat.shape[0]

        logits = self.router(flat)                               # (N, n_routed)
        probs = F.softmax(logits, dim=-1)
        top_w, top_i = probs.topk(self.top_k, dim=-1)             # (N, top_k) each
        top_w = top_w / top_w.sum(dim=-1, keepdim=True)           # renormalize among selected

        out = torch.zeros_like(flat)
        for e, expert in enumerate(self.routed_experts):
            token_slot, k_slot = (top_i == e).nonzero(as_tuple=True)   # which tokens picked expert e, and at which of their top_k slots
            if token_slot.numel() == 0:
                continue
            expert_out = expert(flat[token_slot])                       # only THIS expert's assigned tokens, real compute saving
            out.index_add_(0, token_slot, expert_out * top_w[token_slot, k_slot].unsqueeze(-1))

        for expert in self.shared_experts:
            out = out + expert(flat)                                    # always on, every token

        if self.training:
            # Switch-Transformer style load-balancing aux loss: encourages the router
            # to spread tokens evenly across experts rather than collapsing onto a few.
            with torch.no_grad():
                assign_count = torch.zeros(self.n_routed, device=x.device)
                assign_count.scatter_add_(0, top_i.reshape(-1), torch.ones_like(top_i.reshape(-1), dtype=torch.float))
                frac_tokens = assign_count / (N * self.top_k)
            frac_prob = probs.mean(dim=0)
            self.last_aux_loss = self.aux_loss_coef * self.n_routed * (frac_tokens * frac_prob).sum()
        else:
            self.last_aux_loss = None

        return out.view(B, T, D)

In [ ]:
%%writefile /kaggle/working/mla_dsa.py
import torch
import torch.nn as nn
import torch.nn.functional as F

from mla import MLA
from dsa import Indexer, chunked_sparse_attention, indexer_distillation_loss


class MLADSA(nn.Module):
    def __init__(self, d_model, n_head, head_dim, d_c, d_c_q, d_rotate,
                 n_idx_heads, idx_head_dim, chunk_size, topk_chunks, forced_local_chunks,
                 q_block=128, indexer_distill_sample=256, dropout=0.0, rope_theta=10000.0):
        super().__init__()
        self.mla = MLA(d_model, n_head, head_dim, d_c, d_c_q, d_rotate, dropout, rope_theta)
        self.indexer = Indexer(d_c_q, d_c, n_idx_heads, idx_head_dim, chunk_size, rope_theta)
        self.chunk_size, self.topk_chunks = chunk_size, topk_chunks
        self.forced_local_chunks, self.q_block = forced_local_chunks, q_block
        self.indexer_distill_sample = indexer_distill_sample

    def forward(self, x, past_kv=None, use_cache=False, sparse=True):
        B, T, _ = x.shape
        past_len = past_kv[0].shape[1] if past_kv is not None else 0

        c_kv, c_q, k_R = self.mla.project(x)
        if past_kv is not None:
            full_c_kv = torch.cat([past_kv[0], c_kv], dim=1)
            full_k_R = torch.cat([past_kv[1], k_R], dim=1)
        else:
            full_c_kv, full_k_R = c_kv, k_R

        Q, _, _ = self.mla.up_project(c_kv, c_q, k_R, past_len, x.device, x.dtype)
        _, K_full, V_full = self.mla._up_project_kv_only(full_c_kv, full_k_R, x.device, x.dtype)

        self.last_indexer_loss = None
        if sparse:
            idx_scores = self.indexer(c_q, full_c_kv, past_len)          # (B, T, past_len+T)
            out = chunked_sparse_attention(
                Q, K_full, V_full, idx_scores, self.chunk_size, self.topk_chunks,
                self.forced_local_chunks, self.q_block, causal_offset=past_len,
            )
            if self.training:
                self.last_indexer_loss = indexer_distillation_loss(
                    Q, K_full, idx_scores, self.chunk_size, causal_offset=past_len,
                    sample_size=self.indexer_distill_sample,
                )
        else:
            total_len = past_len + T
            q_pos = torch.arange(past_len, total_len, device=x.device).unsqueeze(1)
            k_pos = torch.arange(0, total_len, device=x.device).unsqueeze(0)
            # Q.dtype, not x.dtype -- same fix as mla.py, same reason (x is pre-projection
            # and can be fp32 under autocast even when Q is correctly fp16 post-projection)
            bias = (k_pos > q_pos).to(Q.dtype) * torch.finfo(Q.dtype).min
            out = F.scaled_dot_product_attention(Q, K_full, V_full, attn_mask=bias[None, None])

        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        out = self.mla.W_O(out)
        present = (full_c_kv, full_k_R) if use_cache else None
        return out, present

In [ ]:
%%writefile /kaggle/working/model.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as torch_checkpoint
from transformers import PretrainedConfig, PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

from mla_dsa import MLADSA
from moe import MoE


class LegalMLADSAConfig(PretrainedConfig):
    model_type = "legal_mla_dsa"

    def __init__(
        self,
        vocab_size=50257, n_positions=4096, n_embd=1024, n_layer=8, n_head=16, head_dim=64,
        d_c=128, d_c_q=192, d_rotate=32,
        n_idx_heads=8, idx_head_dim=32, chunk_size=16, topk_chunks=64, forced_local_chunks=2, q_block=128,
        indexer_distill_sample=256,
        n_routed_experts=6, n_shared_experts=1, moe_top_k=2, expert_d_ff=896, moe_aux_coef=0.01,
        indexer_loss_coef=0.1,
        dropout=0.0, initializer_range=0.02, rms_norm_eps=1e-5,
        bos_token_id=50256, eos_token_id=50256, pad_token_id=50256,
        activation_checkpointing=False, tie_word_embeddings=True,
        **kwargs,
    ):
        self.vocab_size, self.n_positions, self.n_embd = vocab_size, n_positions, n_embd
        self.n_layer, self.n_head, self.head_dim = n_layer, n_head, head_dim
        self.d_c, self.d_c_q, self.d_rotate = d_c, d_c_q, d_rotate
        self.n_idx_heads, self.idx_head_dim = n_idx_heads, idx_head_dim
        self.chunk_size, self.topk_chunks, self.forced_local_chunks, self.q_block = \
            chunk_size, topk_chunks, forced_local_chunks, q_block
        self.indexer_distill_sample = indexer_distill_sample
        self.n_routed_experts, self.n_shared_experts, self.moe_top_k, self.expert_d_ff = \
            n_routed_experts, n_shared_experts, moe_top_k, expert_d_ff
        self.moe_aux_coef, self.indexer_loss_coef = moe_aux_coef, indexer_loss_coef
        self.dropout, self.initializer_range, self.rms_norm_eps = dropout, initializer_range, rms_norm_eps
        self.use_cache = True
        self.activation_checkpointing = activation_checkpointing
        super().__init__(
            bos_token_id=bos_token_id, eos_token_id=eos_token_id, pad_token_id=pad_token_id,
            tie_word_embeddings=tie_word_embeddings, **kwargs,
        )


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


class DecoderBlock(nn.Module):
    def __init__(self, config: LegalMLADSAConfig):
        super().__init__()
        self.ln_1 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.attn = MLADSA(
            config.n_embd, config.n_head, config.head_dim, config.d_c, config.d_c_q, config.d_rotate,
            config.n_idx_heads, config.idx_head_dim, config.chunk_size, config.topk_chunks,
            config.forced_local_chunks, config.q_block, config.indexer_distill_sample, config.dropout,
        )
        self.ln_2 = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.moe = MoE(config.n_embd, config.n_routed_experts, config.n_shared_experts,
                        config.moe_top_k, config.expert_d_ff, config.moe_aux_coef)

    def forward(self, x, past_kv=None, use_cache=False, sparse=True):
        attn_out, present = self.attn(self.ln_1(x), past_kv, use_cache, sparse)
        x = x + attn_out
        x = x + self.moe(self.ln_2(x))
        return x, present, self.attn.last_indexer_loss, self.moe.last_aux_loss


class LegalMLADSAForCausalLM(PreTrainedModel):
    config_class = LegalMLADSAConfig
    _tied_weights_keys = {"lm_head.weight": "wte.weight"}

    def __init__(self, config: LegalMLADSAConfig):
        super().__init__(config)
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.ModuleList([DecoderBlock(config) for _ in range(config.n_layer)])
        self.ln_f = RMSNorm(config.n_embd, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.wte.weight
        self.post_init()

    def _init_weights(self, module):
        std = self.config.initializer_range
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
        for name, p in module.named_parameters(recurse=False):
            if name in ("W_O.weight", "down_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * self.config.n_layer))

    def get_input_embeddings(self):
        return self.wte

    def set_input_embeddings(self, value):
        self.wte = value

    def get_output_embeddings(self):
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        self.lm_head = new_embeddings

    def forward(self, input_ids, attention_mask=None, past_key_values=None, labels=None, use_cache=None):
        # fp16 autocast set HERE, not left to Trainer -- under nn.DataParallel the
        # matmuls measurably ran in fp32 otherwise (model 1's finding). Only for
        # unpadded batches; generation's padded path keeps exact fp32.
        amp = input_ids.is_cuda and (attention_mask is None or bool(attention_mask.bool().all()))
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):
            return self._forward_impl(input_ids, attention_mask, past_key_values, labels, use_cache)
        # NOTE: no **kwargs param above on purpose -- Trainer treats a forward()
        # that accepts **kwargs as self-normalizing its loss, then multiplies by
        # n_gpu and skips the grad-accumulation division (model 1's loss-scale bug).

    def _forward_impl(self, input_ids, attention_mask, past_key_values, labels, use_cache):
        use_cache = self.config.use_cache if use_cache is None else use_cache
        if self.training:
            use_cache = False
        past_len = past_key_values[0][0].shape[1] if past_key_values is not None else 0
        total_len = past_len + input_ids.shape[1]
        # decide sparse/dense from the TOTAL context (cache included), not just this
        # call's input length -- otherwise single-token generation steps always see
        # length 1 and silently fall back to dense every time, even with a long cache
        sparse = total_len > self.config.chunk_size * 2

        x = self.drop(self.wte(input_ids))
        presents = [] if use_cache else None
        indexer_losses, moe_losses = [], []
        for i, block in enumerate(self.blocks):
            pkv = past_key_values[i] if past_key_values is not None else None
            if self.training and self.config.activation_checkpointing:
                def run(h, blk=block, pkv=pkv):
                    h2, _, idx_l, moe_l = blk(h, pkv, False, sparse)
                    if idx_l is not None:
                        indexer_losses.append(idx_l)
                    if moe_l is not None:
                        moe_losses.append(moe_l)
                    return h2
                x = torch_checkpoint(run, x, use_reentrant=False)
            else:
                x, present, idx_l, moe_l = block(x, pkv, use_cache, sparse)
                if use_cache:
                    presents.append(present)
                if idx_l is not None:
                    indexer_losses.append(idx_l)
                if moe_l is not None:
                    moe_losses.append(moe_l)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), ignore_index=-100,
            )
            if indexer_losses:
                loss = loss + self.config.indexer_loss_coef * torch.stack(indexer_losses).mean()
            if moe_losses:
                loss = loss + torch.stack(moe_losses).mean()

        return CausalLMOutputWithPast(
            loss=loss,
            logits=None if labels is not None else logits,
            past_key_values=tuple(presents) if use_cache else None,
        )

In [ ]:
%%writefile /kaggle/working/generate_utils.py
"""
Drop-in replacement for the starter notebook's generate_batch() in
'## 19. Generate 128 Words'. Same signature and return shape, so cell 20
(the prediction loop) and cells 21-22 (prediction.json / submission.zip)
need no changes at all.

Written as a plain per-step loop against the model's own forward() and its
tuple-based KV cache (see legal_gqa_model.py) rather than model.generate(),
to avoid depending on transformers' internal Cache-class machinery.
"""
import torch
import torch.nn.functional as F


def _top_p_filter(logits, top_p):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    probs = F.softmax(sorted_logits, dim=-1)
    cum_probs = torch.cumsum(probs, dim=-1)
    # keep the smallest prefix whose cumulative prob exceeds top_p;
    # always keep at least the top-1 token
    remove = cum_probs > top_p
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits[remove] = float("-inf")
    out = torch.full_like(logits, float("-inf"))
    out.scatter_(-1, sorted_idx, sorted_logits)
    return out


def _build_word_token_table(tokenizer):
    """token_id -> lowercased word it produces, only for tokens that start a
    new word (GPT-2 byte-level BPE marks these with a leading space)."""
    table = {}
    for tid in range(len(tokenizer)):
        text = tokenizer.decode([tid])
        if text[:1] in (" ", "\n"):
            word = text.strip().lower()
            if word:
                table.setdefault(word, []).append(tid)
    return {w: torch.tensor(ids) for w, ids in table.items()}


def _verbatim_overlap(prompt_words, continuation_words, min_run=6):
    """True if a >= min_run-word run from the continuation appears verbatim in
    the prompt (case-insensitive). Used only for a post-hoc visibility check --
    the actual prevention happens via trigram seeding below."""
    prompt_joined = " ".join(w.lower() for w in prompt_words)
    for i in range(len(continuation_words) - min_run + 1):
        window = " ".join(w.lower() for w in continuation_words[i:i + min_run])
        if window and window in prompt_joined:
            return window
    return None


@torch.inference_mode()
def generate_batch(
    prompts=None,
    model=None,
    tokenizer=None,
    device=None,
    target_words=128,
    max_new_tokens=384,
    temperature=0.7,
    top_p=0.95,
    tail_window=8,  # tokens looked at when checking for a repeated word
    token_ids=None,  # optional: list of lists of ints (one per prompt) -- skips re-tokenizing
):
    model.eval()
    max_length = model.config.n_positions - max_new_tokens
    if token_ids is not None:
        trunc = [ids[:max_length] for ids in token_ids]  # same right-truncation as the tokenizer path below
        pad_id = tokenizer.pad_token_id
        B = len(trunc)
        maxlen = max(len(ids) for ids in trunc)
        input_ids = torch.full((B, maxlen), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((B, maxlen), dtype=torch.long)
        for i, ids in enumerate(trunc):
            input_ids[i, maxlen - len(ids):] = torch.tensor(ids, dtype=torch.long)  # left-pad, matches tokenizer.padding_side="left"
            attention_mask[i, maxlen - len(ids):] = 1
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
    else:
        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True,
            max_length=max_length,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]
    input_lengths = attention_mask.sum(dim=1)
    B = input_ids.size(0)

    word_table = getattr(generate_batch, "_word_table_cache", None)
    if word_table is None:
        word_table = _build_word_token_table(tokenizer)
        generate_batch._word_table_cache = word_table  # cache across calls

    # prefill: run the full prompt once to seed the KV cache
    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True)
    past_key_values = out.past_key_values
    next_logits = out.logits[:, -1, :]

    # Real per-batch generation budget, not the fixed max_new_tokens guess -- a fixed
    # cap under-provisions short prompts (which have far more safe room than that) and
    # can still be too little for hard rows where repetition-blocking wastes steps.
    # Bounded by the model's actual trained context length, so this never pushes
    # generation into positions the model was never trained on.
    available_steps = max(1, model.config.n_positions - input_ids.shape[1])

    generated = [[] for _ in range(B)]
    trigram_seen = [set() for _ in range(B)]
    # Seed with the PROMPT's own trigrams too, not just trigrams generated so far --
    # this makes the SAME "block recurrence of any seen trigram" guard also stop the
    # model from reproducing 3+ consecutive tokens verbatim from the prompt (the task's
    # "must not copy the prompt" requirement), reusing an already-tested mechanism
    # rather than adding a new one.
    prompt_ids_per_row = []
    for row in range(B):
        real_ids = input_ids[row][attention_mask[row].bool()].tolist()
        prompt_ids_per_row.append(real_ids)
        for i in range(len(real_ids) - 2):
            trigram_seen[row].add(tuple(real_ids[i:i + 3]))
    finished_word_count = [0] * B
    cur_ids = input_ids
    cur_mask = attention_mask

    for _ in range(available_steps):
        step_logits = next_logits / max(temperature, 1e-5)

        for row in range(B):
            if finished_word_count[row] >= target_words:
                continue

            # --- guard 0: block natural EOS until the target is reached ---
            # Without this, a model trained on EOS-separated documents (exactly
            # what every model in this project is trained on) will sometimes
            # naturally choose EOS mid-continuation when it "feels" a document
            # should end. That token still consumes one of the available_steps
            # loop iterations and gets appended to generated[row], but
            # tokenizer.decode(..., skip_special_tokens=True) at the very end
            # silently strips it -- so it contributes zero words while still
            # spending real budget. Confirmed as the actual mechanism behind a
            # real shortfall (min=74/128 on a production run), not the
            # not-enough-context-room case the WARNING below is actually for.
            if tokenizer.eos_token_id is not None:
                step_logits[row, tokenizer.eos_token_id] = float("-inf")

            # --- guard 1: block a 3rd consecutive repeat of the same word ---
            tail_ids = torch.tensor(generated[row][-tail_window:])
            if tail_ids.numel() > 0:
                tail_text = tokenizer.decode(tail_ids, skip_special_tokens=True)
                words = tail_text.split()
                if len(words) >= 2 and words[-1].lower() == words[-2].lower():
                    block_ids = word_table.get(words[-1].lower())
                    if block_ids is not None:
                        step_logits[row, block_ids.to(device)] = float("-inf")

        # --- guard 2: block recurrence of any trigram already generated ---
        # checked only against each row's current top-50 candidates (the
        # tokens sampling would realistically pick anyway) rather than the
        # full vocab, since this runs every generation step.
        for row in range(B):
            if finished_word_count[row] >= target_words or len(generated[row]) < 2:
                continue
            prefix = (generated[row][-2], generated[row][-1])
            seen = trigram_seen[row]
            # only tokens that would recreate a seen trigram get blocked;
            # we look this up lazily per-row since the seen set is small
            for tid in torch.topk(step_logits[row], k=min(50, step_logits.size(-1))).indices.tolist():
                if (prefix[0], prefix[1], tid) in seen:
                    step_logits[row, tid] = float("-inf")

        filtered = _top_p_filter(step_logits, top_p)
        probs = F.softmax(filtered, dim=-1)
        next_tokens = torch.multinomial(probs, num_samples=1).squeeze(-1)

        for row in range(B):
            if finished_word_count[row] >= target_words:
                next_tokens[row] = tokenizer.eos_token_id
                continue
            tok = next_tokens[row].item()
            generated[row].append(tok)
            if len(generated[row]) >= 3:
                trigram_seen[row].add(tuple(generated[row][-3:]))
            text_so_far = tokenizer.decode(generated[row], skip_special_tokens=True)
            finished_word_count[row] = len(text_so_far.split())

        if all(c >= target_words for c in finished_word_count):
            break

        next_col = next_tokens.unsqueeze(1)
        cur_mask = torch.cat([cur_mask, torch.ones(B, 1, dtype=cur_mask.dtype, device=device)], dim=1)
        out = model(input_ids=next_col, attention_mask=cur_mask, past_key_values=past_key_values, use_cache=True)
        past_key_values = out.past_key_values
        next_logits = out.logits[:, -1, :]

    predictions = []
    for row in range(B):
        text = tokenizer.decode(generated[row], skip_special_tokens=True)
        words = text.split()[:target_words]
        if len(words) < target_words:
            # only happens if this row's prompt was long enough that even the FULL
            # available context budget (n_positions - prompt_length) wasn't enough to
            # reach target_words -- a real limit for the longest prompts, not a bug;
            # printed so it's visible rather than silently submitting a short prediction
            print(f"WARNING: row {row} only reached {len(words)}/{target_words} words "
                  f"(prompt used {input_ids.shape[1]} of {model.config.n_positions} context tokens)")

        prompt_text = tokenizer.decode(prompt_ids_per_row[row], skip_special_tokens=True)
        overlap = _verbatim_overlap(prompt_text.split(), words)
        if overlap is not None:
            # trigram seeding above should make this rare -- printed so it's visible
            # rather than silently submitting a continuation that copies the prompt
            print(f"WARNING: row {row} continuation contains a run copied from the "
                  f"prompt: \"{overlap}\"")

        predictions.append(" ".join(words))
    return predictions

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working")
from model import LegalMLADSAConfig, LegalMLADSAForCausalLM
from generate_utils import generate_batch

## 7. Load The Trained Model

In [ ]:
ckpt_dirs = find_dirs("model.safetensors")
final_dirs = [d for d in ckpt_dirs if d.rstrip("/").endswith("legal-mladsa-final") and "checkpoint-" not in d]
if not final_dirs:
    print("No 'legal-mladsa-final' folder found. Candidates seen:")
    for d in ckpt_dirs:
        print(" ", d)
    raise FileNotFoundError("Attach model 2's training notebook output (Add Input).")
MODEL_DIR = final_dirs[0]
print("Loading model from:", MODEL_DIR)

model = LegalMLADSAForCausalLM.from_pretrained(MODEL_DIR).to(device).eval()
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## 8. Generate Predictions for the Test Set

Uses `generate_batch` as-is: top-p + temperature sampling, blocks a 3rd consecutive repeat of the same word, blocks trigram recurrence, and -- seeded with the prompt's own trigrams -- prevents the continuation from copying the prompt verbatim (verified in testing: zero copy warnings across 30 trials on a deliberately adversarial, repetitive prompt). Uses the pre-tokenized cache above if attached, otherwise tokenizes from raw text. Guarantees every continuation reaches exactly 128 words when the prompt's length leaves enough context room to do so.

In [ ]:
predictions = {}
items = list(test_data.items())
total = len(items)
t0 = time.time()

for start in range(0, total, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    ids = [str(eid) for eid, _ in batch]

    if test_token_ids is not None:
        gen_kwargs = {"token_ids": [test_token_ids[eid] for eid in ids]}
    else:
        gen_kwargs = {"prompts": [rec["text"] for _, rec in batch]}

    outs = generate_batch(
        model=model, tokenizer=tokenizer, device=device,
        target_words=TARGET_GENERATION_WORDS, max_new_tokens=MAX_GENERATION_TOKENS,
        temperature=TEMPERATURE, top_p=TOP_P, **gen_kwargs,
    )
    for eid, text in zip(ids, outs):
        predictions[eid] = {"text": text}

    done = start + len(batch)
    elapsed = time.time() - t0
    rate = done / elapsed
    eta = (total - done) / rate if rate > 0 else float("inf")
    print(f"{done}/{total} ({100*done/total:.1f}%) | {elapsed/60:.1f} min elapsed | ETA {eta/60:.1f} min")

## 9. Save prediction.json and Create Submission ZIP

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print("Saved:", PREDICTION_PATH)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")
print("Codabench submission created:", SUBMISSION_ZIP)

lens = [len(v["text"].split()) for v in predictions.values()]
print(f"word count: min={min(lens)} max={max(lens)} mean={sum(lens)/len(lens):.1f}")
print("any empty predictions:", sum(1 for l in lens if l == 0))